# LaminDB — User Journey

LaminDB is the storage layer. The query interface is the PyArrow dataset API, accessed directly via `collection.open()` — no caching, no extra setup.

**Steps:** access → query → filtered query → append → evolve schema

In [ ]:
import time
from contextlib import contextmanager

timings = {}

@contextmanager
def bench(step):
    """Time a step and record it for the final benchmark table."""
    t0 = time.perf_counter()
    yield
    timings[step] = time.perf_counter() - t0
    print(f"  {step}: {timings[step]:.3f}s")

## 1. Access

`collection.open()` streams the Parquet shards from S3 as one PyArrow dataset.

In [ ]:
import lamindb as ln
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc

ln.track(path="lamin_pipeline.ipynb", project="Lakehouse benchmarks v1")
collection = ln.Collection.get("K6X8Ejk3fjgAZT6h0000")

with bench("access"):
    dataset = collection.open()
    _count = dataset.count_rows()

print(f"Total rows: {_count:,}")


## 2. Query — per-sample stats and recurrent regions

In [ ]:
def calculate_sample_stats(arrow_table):
    """Summary statistics per sample."""
    df = arrow_table.to_pandas()
    rows = []
    for name in df["SAMPLE_NAME"].unique():
        s = df[df["SAMPLE_NAME"] == name]
        dels = s[s["INFO_SVLEN"] < 0]
        rows.append({
            "Sample": name,
            "Total_CNVs": len(s),
            "Deletions": len(dels),
            "Median_Deletion_Size": abs(dels["INFO_SVLEN"].median()) if not dels.empty else 0,
            "Homozygous_CNVs": (s["SAMPLE_GT"] == "1/1").sum(),
            "Heterozygous_CNVs": (s["SAMPLE_GT"] == "0/1").sum(),
        })
    return pd.DataFrame(rows)

def identify_recurrent_regions(arrow_table, proximity=1000):
    """Regions with recurrent CNVs across >=2 samples."""
    df = arrow_table.select(["CHROM", "POS", "SAMPLE_NAME"]).to_pandas()
    df["region_key"] = df["CHROM"] + ":" + ((df["POS"] // proximity) * proximity).astype(str)
    counts = df.groupby("region_key")["SAMPLE_NAME"].nunique()
    return counts[counts >= 2]

In [ ]:
with bench("query_stats"):
    stats_df = calculate_sample_stats(dataset.to_table())
stats_df.head()

In [ ]:
with bench("query_recurrent"):
    recurrent = identify_recurrent_regions(dataset.to_table())
print(f"Identified {len(recurrent)} recurrent regions.")

## 3. Filtered query — point query via PyArrow compute pushdown

In [ ]:
with bench("filtered_query"):
    filtered = dataset.filter(
        (pc.field("CHROM") == "1")
        & (pc.field("POS") >= 1_000_000)
        & (pc.field("POS") <= 50_000_000)
    ).to_table()
print(f"Variants in chr1:1M-50M: {filtered.num_rows}")

## 4. Append — schema-validated, lineage tracked

LaminDB validates the new artifact against the registered schema and records lineage automatically.

In [ ]:
import tempfile, pathlib, pyarrow.parquet as pq

schema_ln = ln.Schema.get(name="1000 Genomes CNV VCF")
_tmp = pathlib.Path(tempfile.mktemp(suffix=".parquet"))

with bench("append"):
    pq.write_table(dataset.to_table().slice(0, 10), _tmp)
    new_artifact = ln.Artifact(
        _tmp,
        description="New CNV samples batch 2 (benchmark stub)",
        schema=schema_ln,
    )
    new_artifact.save()
    _tmp.unlink(missing_ok=True)

print(f"Saved artifact uid={new_artifact.uid}")
new_artifact.view_lineage()


## 5. Evolve schema — update the registered schema

Subsequent artifacts saved against this schema are validated against the new definition.

In [ ]:
with bench("evolve_schema"):
    qc_feat, _ = ln.Feature.objects.get_or_create(name="QC_PASS", dtype="bool")
    schema_ln.add_optional_features([qc_feat])
print("Schema updated with QC_PASS")


## Benchmark summary

In [ ]:
import pandas as pd
pd.DataFrame(
    [{"step": k, "seconds": round(v, 3)} for k, v in timings.items()]
)

In [ ]:
try:
    ln.finish()
except Exception:
    pass
